# ✈️ Обнаружение самолётов — обучение на Colab

**Порядок запуска:** выполняй ячейки сверху вниз по одной.

Перед запуском убедись что выбран GPU:  
`Среда выполнения → Сменить среду выполнения → T4 GPU`

## 1. Проверка GPU

In [ ]:
import torch

print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU не найден! Зайди в Среда выполнения → Сменить среду → T4 GPU")

## 2. Монтирование Google Drive

Чекпоинты будут сохраняться на Drive — не потеряются при срыве сессии.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/airplane-detection'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Папка на Drive: {DRIVE_DIR}")

## 3. Клонирование репозитория

⚠️ **Замени URL на свой репозиторий!**

In [ ]:
REPO_URL = 'https://github.com/ВАШ_ЮЗЕРНЕЙМ/airplane-detection.git'  # <-- замени!

%cd /content
!git clone {REPO_URL}
%cd airplane-detection
!ls

## 4. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt
!apt-get install -y -q p7zip-full
print("✅ Зависимости установлены")

## 5. Скачивание и распаковка датасета

HRPlanes с Zenodo (~10 GB). Займёт около 5-10 минут.

In [ ]:
!python scripts/download.py

In [ ]:
# Проверяем что датасет лежит правильно
from pathlib import Path

for split in ('train', 'val', 'test'):
    imgs = list(Path(f'data/hrplanes/{split}/images').glob('*.jpg'))
    lbls = list(Path(f'data/hrplanes/{split}/labels').glob('*.txt'))
    print(f"{split:5}: {len(imgs):5} изображений, {len(lbls):5} меток")

## 6. Конфигурация путей чекпоинтов

Перенаправляем сохранение весов на Google Drive.

In [ ]:
import os

# Создаём символическую ссылку: assets/models -> папка на Drive
models_dir = f'{DRIVE_DIR}/models'
os.makedirs(models_dir, exist_ok=True)
os.makedirs(f'{models_dir}/epochs', exist_ok=True)

local_models = Path('/content/airplane-detection/assets/models')
if local_models.exists() and not local_models.is_symlink():
    import shutil
    shutil.rmtree(local_models)

if not local_models.is_symlink():
    os.symlink(models_dir, str(local_models))

print(f"✅ Чекпоинты будут сохраняться в: {models_dir}")
print("   Структура:")
print("   assets/models/best.pt          — лучший по mAP50")
print("   assets/models/last.pt          — последняя эпоха")
print("   assets/models/epochs/epoch_NNN.pt — каждая эпоха")

## 7. Smoke-test — 3 эпохи

Запускаем сначала 3 эпохи чтобы убедиться что всё работает (~5 мин).

In [ ]:
import yaml

# Временно ставим 3 эпохи для smoke-test
with open('configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg_smoke = cfg.copy()
cfg_smoke['training'] = dict(cfg['training'])
cfg_smoke['training']['epochs'] = 3

with open('configs/config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_smoke, f)

print("Конфиг для smoke-test сохранён: configs/config_smoke.yaml")
print(f"Эпох: {cfg_smoke['training']['epochs']}")

In [ ]:
!python scripts/train.py --config configs/config_smoke.yaml

## 8. Полное обучение

Если smoke-test прошёл успешно — запускаем полное обучение.

**Рекомендация:** поставь `epochs: 60` — гарантированно влезет в 2 часа на T4.

In [ ]:
# Можно изменить количество эпох здесь
EPOCHS = 60  # рекомендуется 60 для 2 часов; поставь 100 если времени больше

with open('configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

cfg['training']['epochs'] = EPOCHS

with open('configs/config_full.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(f"Запускаем обучение на {EPOCHS} эпох...")

In [ ]:
!python scripts/train.py --config configs/config_full.yaml

## 9. Оценка лучшего чекпоинта

In [ ]:
!python scripts/val.py --config configs/config_full.yaml --ckpt assets/models/best.pt

In [ ]:
!python scripts/test.py --config configs/config_full.yaml --ckpt assets/models/best.pt

## 10. Список сохранённых чекпоинтов

In [ ]:
import os
from pathlib import Path

models_path = Path('assets/models')
print("Чекпоинты на Google Drive:")
for f in sorted(models_path.rglob('*.pt')):
    size_mb = f.stat().st_size / 1e6
    print(f"  {str(f):<45} {size_mb:.1f} MB")